# 02 — Limpeza

Transforma o HTML bruto de cada política em texto corrido, só com o conteúdo do documento. Nada em `data/raw/` é alterado; os textos vão para `data/interim/politicas/`.

As funções estão em `src/limpeza.py`.

In [1]:
from pathlib import Path
import os
import sys

# Os caminhos do projeto são relativos à raiz do repositório.
RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

import pandas as pd

pd.set_option("display.max_colwidth", 90)

## O que a limpeza faz

1. Detecta a codificação a partir dos bytes do arquivo.
2. Descarta scripts, estilos, menus, cabeçalhos, rodapés e formulários.
3. Escolhe a área de conteúdo da página e a converte em texto, uma linha por bloco, marcando os títulos com `#`.
4. Quando a página quase não tem texto, procura o documento embutido em um script (caso da Emotiv).
5. Aplica os **recortes**: em páginas que reúnem vários documentos ou trazem menus dentro da área principal, guarda só o trecho da política.

In [2]:
from src.limpeza import RECORTES, limpar_politicas

resumo = pd.DataFrame(limpar_politicas())
resumo[["id_documento", "idioma", "caracteres", "palavras", "recorte"]]

,id_documento,idioma,caracteres,palavras,recorte
0,P01-1,en,44254,7036,sim
1,P01-2,en,15544,2464,nao
2,P02-1,en,10905,1796,sim
3,P03-1,en,44471,6899,nao
4,P03-2,en,3832,601,sim
5,P04-1,en,13921,2210,nao
6,P05-1,en,13147,1739,sim
7,P06-1,zh,5125,4515,nao
8,P07-1,en,26051,4157,nao
9,P07-2,en,1546,269,nao


## Recortes aplicados

Cada recorte é definido por um marcador de início e um de fim. Se um marcador deixa de existir — porque o site mudou de estrutura —, a limpeza falha com erro, em vez de gerar uma política incompleta.

In [3]:
pd.DataFrame(
    [(doc, inicio, fim or "(até o fim)") for doc, (inicio, fim) in RECORTES.items()],
    columns=["id_documento", "começa em", "termina antes de"],
)

,id_documento,começa em,termina antes de
0,P01-1,# Privacy Policy,# Hardware Warnings
1,P02-1,# Privacy Policy,(até o fim)
2,P03-2,# EMOTIV Consumer Health Data Privacy Policy,(até o fim)
3,P05-1,# Privacy Policy,Order Now
4,P09-1,INTRODUCTION TO NEUPHONY,# New from Neuphony\?
5,P13-1,Privacy Policy and Terms of Use,TERMS OF USE


## Conferência

Primeira e última linha de cada texto, para confirmar que começam no título da política e não terminam em rodapé.

In [4]:
from src.limpeza import PASTA_INTERIM

def pontas(arquivo):
    linhas = (PASTA_INTERIM / arquivo).read_text(encoding="utf-8").splitlines()
    return pd.Series({"primeira linha": linhas[0][:70], "última linha": linhas[-1][:70]})

resumo.set_index("id_documento")["arquivo_texto"].apply(pontas)

,primeira linha,última linha
id_documento,,
P01-1,# Privacy Policy,PrivacyEU@choosemuse.com
P01-2,# Privacy policy,Should you have any questions about our privacy practices or this Priv
P02-1,# Privacy Policy,"Last updated May 25, 2009"
P03-1,# EMOTIV Privacy Policy,"By (i) clicking “I accept EMOTIV’s Privacy Policy” or its equivalent,"
P03-2,# EMOTIV Consumer Health Data Privacy Policy,We do not allow third parties to collect your Consumer Health Data thr
P04-1,# Privacy policy,We will notify you before we make material changes to this policy and
P05-1,# Privacy Policy,+1-888-979-2724
P06-1,# 深圳市宏智力科技有限公司隐私政策,我们会定期评估本协议的遵守情况。如果我们收到了您的正式书面投诉，我们会尝试与您直接联系解决。我们会配合国家有关部门，包括数据保护部门，解决您
P07-1,# Privacy Policy,http://neurosity.co/contact


In [5]:
fontes = pd.read_csv("data/raw/fontes_politicas.csv")
resumo.merge(fontes[["id_documento", "tipo"]]).groupby(["tipo", "idioma"])["palavras"].agg(["count", "median", "min", "max"])

count  median   min   max
tipo         idioma                           
complementar en          6   466.5   269  2464
principal    en         14  3562.0  1055  7036
             zh          1  4515.0  4515  4515

## Limitações da limpeza

- A política da Macrotellect está apenas em chinês e foi mantida no idioma original; a tradução fica para a etapa de avaliação.
- Alguns textos conservam poucas linhas de navegação no início ou no fim (por exemplo, links de rodapé), que não afetam a leitura.
- A limpeza guarda o texto, não a formatação: tabelas viram linhas soltas.